# RECI — Entrenamiento MobileNetV2
**Sistema Experto de Reciclaje — PUCE Manabí**

Este notebook entrena un modelo MobileNetV2 con transfer learning
y lo exporta como `.tflite` listo para usar en el sistema RECI.

### Antes de empezar:
1. Ir a **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU**
2. Ejecutar las celdas en orden de arriba hacia abajo

## Paso 1 — Conectar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive conectado correctamente')

Mounted at /content/drive
Drive conectado correctamente


In [ ]:
import os

print("Contenido de separados:")
for item in os.listdir('/content/drive/MyDrive/data set axel 1/separados'):
    print(f"  {item}")

Contenido de separados:
  plastico
  vidrio
  lata
  organico


## Paso 2 — Verificar que las fotos están en la ruta correcta

In [ ]:
import os

# Rutas del dataset propio RECI
RUTA_PLASTICO = '/content/drive/MyDrive/RECI_dataset_propio/plastico'
RUTA_VIDRIO   = '/content/drive/MyDrive/RECI_dataset_propio/vidrio'

# Verificar que existen y contar fotos
fotos_plastico = len(os.listdir(RUTA_PLASTICO))
fotos_vidrio   = len(os.listdir(RUTA_VIDRIO))

print(f'Plastico : {fotos_plastico} fotos')
print(f'Vidrio   : {fotos_vidrio} fotos')
print(f'Total    : {fotos_plastico + fotos_vidrio} fotos')

Plastico : 2807 fotos
Vidrio   : 1087 fotos
Total    : 3894 fotos


## Paso 3 — Organizar dataset en estructura estándar

In [ ]:
import shutil
import random
import os

RUTA_PLASTICO = '/content/drive/MyDrive/RECI_dataset_propio/plastico'
RUTA_VIDRIO   = '/content/drive/MyDrive/RECI_dataset_propio/vidrio'
DATASET_DIR   = '/content/drive/MyDrive/RECI_dataset_propio/dataset_organizado'
CLASES        = {'plastico': RUTA_PLASTICO, 'vidrio': RUTA_VIDRIO}
SPLIT_TRAIN   = 0.85

for split in ['train', 'val']:
    for clase in CLASES:
        os.makedirs(f'{DATASET_DIR}/{split}/{clase}', exist_ok=True)

for clase, ruta_origen in CLASES.items():
    # Ver qué fotos ya están en train y val
    ya_en_train = set(os.listdir(f'{DATASET_DIR}/train/{clase}'))
    ya_en_val   = set(os.listdir(f'{DATASET_DIR}/val/{clase}'))
    ya_copiadas = ya_en_train | ya_en_val

    # Solo procesar fotos nuevas
    todas = [f for f in os.listdir(ruta_origen)
             if f.lower().endswith(('.jpg', '.jpeg', '.png', '.webp'))]
    nuevas = [f for f in todas if f not in ya_copiadas]

    if not nuevas:
        print(f'{clase}: sin fotos nuevas (total: {len(todas)})')
        continue

    random.shuffle(nuevas)
    corte      = int(len(nuevas) * SPLIT_TRAIN)
    train_imgs = nuevas[:corte]
    val_imgs   = nuevas[corte:]

    for img in train_imgs:
        shutil.copy(f'{ruta_origen}/{img}', f'{DATASET_DIR}/train/{clase}/{img}')
    for img in val_imgs:
        shutil.copy(f'{ruta_origen}/{img}', f'{DATASET_DIR}/val/{clase}/{img}')

    total_train = len(os.listdir(f'{DATASET_DIR}/train/{clase}'))
    total_val   = len(os.listdir(f'{DATASET_DIR}/val/{clase}'))
    print(f'{clase}: +{len(nuevas)} nuevas → total {total_train} train, {total_val} val')

print('Dataset actualizado correctamente')

plastico: +2807 nuevas → total 2385 train, 422 val
vidrio: +1087 nuevas → total 923 train, 164 val
Dataset actualizado correctamente


In [ ]:
import os

DATASET_DIR = '/content/drive/MyDrive/data set axel 1/dataset_organizado'

for split in ['train', 'val']:
    for clase in ['plastico', 'vidrio']:
        ruta = f'{DATASET_DIR}/{split}/{clase}'
        if os.path.exists(ruta):
            count = len(os.listdir(ruta))
            print(f'{split}/{clase}: {count} fotos copiadas')
        else:
            print(f'{split}/{clase}: carpeta no creada aún')

train/plastico: 7042 fotos copiadas
train/vidrio: 5671 fotos copiadas
val/plastico: 1243 fotos copiadas
val/vidrio: 1001 fotos copiadas


## Paso 4 — Preparar datos para entrenamiento

In [ ]:
import tensorflow as tf
import os

DATASET_DIR = '/content/drive/MyDrive/RECI_dataset_propio/dataset_organizado'

print(f'TensorFlow version: {tf.__version__}')
print(f'GPU disponible: {tf.config.list_physical_devices("GPU")}')

IMG_SIZE   = 224
BATCH_SIZE = 32

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip('horizontal'),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomBrightness(0.1),
])

train_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATASET_DIR}/train',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    f'{DATASET_DIR}/val',
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

CLASES_NOMBRES = train_ds.class_names
print(f'Clases detectadas: {CLASES_NOMBRES}')

AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.map(lambda x, y: (data_augmentation(x, training=True), y))
train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.prefetch(buffer_size=AUTOTUNE)

TensorFlow version: 2.20.0
GPU disponible: []
Found 3308 files belonging to 2 classes.
Found 586 files belonging to 2 classes.
Clases detectadas: ['plastico', 'vidrio']


## Paso 5 — Construir modelo MobileNetV2

In [ ]:
NUM_CLASES = len(CLASES_NOMBRES)

# Cargar MobileNetV2 preentrenado en ImageNet (sin la capa final)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=False,
    weights='imagenet'
)

# Congelar el modelo base — solo entrenar las capas nuevas
base_model.trainable = False

# Construir modelo completo
inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x       = tf.keras.applications.mobilenet_v2.preprocess_input(inputs)
x       = base_model(x, training=False)
x       = tf.keras.layers.GlobalAveragePooling2D()(x)
x       = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(NUM_CLASES, activation='softmax')(x)

model = tf.keras.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ subtract_1 (Subtract)           │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 2)              │         2,562 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,260,546 (8.62 MB)

 Trainable params: 2,562 (10.01 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

## Paso 6 — Entrenar (Fase 1: capas nuevas)

In [ ]:
RUTA_DRIVE = '/content/drive/MyDrive/RECI_dataset_propio'

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{RUTA_DRIVE}/mejor_modelo.keras',
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print('Entrenando fase 1 (capas nuevas)...')
historia1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

print(f'\nMejor precisión fase 1: {max(historia1.history["val_accuracy"]):.1%}')

Entrenando fase 1 (capas nuevas)...
Epoch 1/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.7479 - loss: 0.5967
Epoch 1: val_accuracy improved from None to 0.94539, saving model to /content/drive/MyDrive/RECI_dataset_propio/mejor_modelo.keras

Epoch 1: finished saving model to /content/drive/MyDrive/RECI_dataset_propio/mejor_modelo.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 305s 3s/step - accuracy: 0.8304 - loss: 0.3963 - val_accuracy: 0.9454 - val_loss: 0.1482
Epoch 2/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9361 - loss: 0.1736
Epoch 2: val_accuracy improved from 0.94539 to 0.97099, saving model to /content/drive/MyDrive/RECI_dataset_propio/mejor_modelo.keras

Epoch 2: finished saving model to /content/drive/MyDrive/RECI_dataset_propio/mejor_modelo.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 275s 3s/step - accuracy: 0.9450 - loss: 0.1523 - val_accuracy: 0.9710 - val_loss: 0.1084
Epoch 3/15
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 2s/step - accuracy: 0.9652 - loss: 0.1057
Epoch 3: val_a

In [ ]:
!nvidia-smi

Thu May 28 18:47:24 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   70C    P0             31W /   70W |    2169MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## Paso 7 — Fine-tuning (Fase 2: afinar el modelo base)

In [ ]:
import tensorflow as tf

RUTA_DRIVE = '/content/drive/MyDrive/RECI_dataset_propio'

try:
    model = tf.keras.models.load_model(f'{RUTA_DRIVE}/mejor_modelo.keras')
    print('Modelo cargado desde Drive correctamente')
except:
    print('Usando modelo de la sesión actual')

base_model = None
for layer in model.layers:
    if 'mobilenetv2' in layer.name.lower():
        base_model = layer
        break

print(f'Base model encontrado: {base_model.name}')

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.00005),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks2 = [
    tf.keras.callbacks.ModelCheckpoint(
        f'{RUTA_DRIVE}/mejor_modelo_ft.keras',
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    ),
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        patience=5,
        restore_best_weights=True,
        verbose=1
    )
]

print('Entrenando fase 2 (fine-tuning)...')
historia2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
    callbacks=callbacks2
)

print(f'\nMejor precisión fase 2: {max(historia2.history["val_accuracy"]):.1%}')

Modelo cargado desde Drive correctamente
Base model encontrado: mobilenetv2_1.00_224
Entrenando fase 2 (fine-tuning)...
Epoch 1/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9582 - loss: 0.1025
Epoch 1: val_accuracy improved from None to 0.99317, saving model to /content/drive/MyDrive/RECI_dataset_propio/mejor_modelo_ft.keras

Epoch 1: finished saving model to /content/drive/MyDrive/RECI_dataset_propio/mejor_modelo_ft.keras
104/104 ━━━━━━━━━━━━━━━━━━━━ 350s 3s/step - accuracy: 0.9764 - loss: 0.0639 - val_accuracy: 0.9932 - val_loss: 0.0357
Epoch 2/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9918 - loss: 0.0243
Epoch 2: val_accuracy did not improve from 0.99317
104/104 ━━━━━━━━━━━━━━━━━━━━ 374s 3s/step - accuracy: 0.9946 - loss: 0.0208 - val_accuracy: 0.9898 - val_loss: 0.0416
Epoch 3/10
104/104 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9965 - loss: 0.0147
Epoch 3: val_accuracy improved from 0.99317 to 0.99659, saving model to /content/drive/MyDrive/RECI_data

## Paso 8 — Evaluar modelo final

In [ ]:
loss, accuracy = model.evaluate(val_ds)
print(f'Precisión final: {accuracy:.1%}')
print(f'Loss final:      {loss:.4f}')

if accuracy >= 0.90:
    print('Modelo listo para producción')
elif accuracy >= 0.80:
    print('Modelo aceptable, considera agregar más fotos para mejorar')
else:
    print('Precisión baja, agregar más fotos o revisar calidad de imágenes')

19/19 ━━━━━━━━━━━━━━━━━━━━ 34s 2s/step - accuracy: 0.9966 - loss: 0.0171
Precisión final: 99.7%
Loss final:      0.0171
Modelo listo para producción


## Paso 9 — Exportar como TFLite

In [ ]:
import os

RUTA_DRIVE = '/content/drive/MyDrive/RECI_dataset_propio'

converter    = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open(f'{RUTA_DRIVE}/model.tflite', 'wb') as f:
    f.write(tflite_model)
with open('/content/model.tflite', 'wb') as f:
    f.write(tflite_model)

with open(f'{RUTA_DRIVE}/labels.txt', 'w') as f:
    for i, clase in enumerate(CLASES_NOMBRES):
        f.write(f'{i} {clase}\n')
with open('/content/labels.txt', 'w') as f:
    for i, clase in enumerate(CLASES_NOMBRES):
        f.write(f'{i} {clase}\n')

print(f'model.tflite : {os.path.getsize("/content/model.tflite") / 1024 / 1024:.1f} MB')
print(f'labels.txt   : {CLASES_NOMBRES}')
print('Guardados en Drive y listos para descargar')

Saved artifact at '/tmp/tmpmiyw6f15'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='input_layer_5')
Output Type:
  TensorSpec(shape=(None, 2), dtype=tf.float32, name=None)
Captures:
  139997526433744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526432976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526436816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526434320: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526436624: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526437968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526435472: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526430672: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526437776: TensorSpec(shape=(), dtype=tf.resource, name=None)
  139997526438736: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1399975264375

## Paso 10 — Descargar archivos al Mac

In [ ]:
from google.colab import files

print('Descargando model.tflite...')
files.download('/content/model.tflite')

print('Descargando labels.txt...')
files.download('/content/labels.txt')

print('Listo — copia ambos archivos a la carpeta model/ de RECI')

Descargando model.tflite...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Descargando labels.txt...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Listo — copia ambos archivos a la carpeta model/ de RECI


## Paso 11 — Actualizar MAPA_CLASES en tm_classifier.py

Después de descargar los archivos, el `labels.txt` va a tener solo 2 clases:
```
0 plastico
1 vidrio
```

El `tm_classifier.py` ya tiene el mapeo para `plastico` y `vidrio` — no hay que cambiar nada en el código.

Solo copia `model.tflite` y `labels.txt` a la carpeta `RECI/model/` y prueba con:
```bash
python3 vision/tm_classifier.py images/prueba7.jpeg
```